In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('Dataset/pdb_data_seq.csv')

In [4]:
df.isnull().sum()

structureId              0
chainId                 10
sequence                28
residueCount             0
macromoleculeType    34817
dtype: int64

In [5]:
df.head()

,structureId,chainId,sequence,residueCount,macromoleculeType
0,100D,A,CCGGCGCCGG,20,DNA/RNA Hybrid
1,100D,B,CCGGCGCCGG,20,DNA/RNA Hybrid
2,101D,A,CGCGAATTCGCG,24,DNA
3,101D,B,CGCGAATTCGCG,24,DNA
4,101M,A,MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDR...,154,Protein


In [7]:
df.shape

(467304, 5)

In [8]:
df.info

<bound method DataFrame.info of        structureId chainId                                           sequence  \
0             100D       A                                         CCGGCGCCGG   
1             100D       B                                         CCGGCGCCGG   
2             101D       A                                       CGCGAATTCGCG   
3             101D       B                                       CGCGAATTCGCG   
4             101M       A  MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDR...   
...            ...     ...                                                ...   
467299        9XIA       A  MNYQPTPEDRFTFGLWTVGWQGRDPFGDATRRALDPVESVQRLAEL...   
467300        9XIM       A  SVQATREDKFSFGLWTVGWQARDAFGDATRTALDPVEAVHKLAEIG...   
467301        9XIM       B  SVQATREDKFSFGLWTVGWQARDAFGDATRTALDPVEAVHKLAEIG...   
467302        9XIM       C  SVQATREDKFSFGLWTVGWQARDAFGDATRTALDPVEAVHKLAEIG...   
467303        9XIM       D  SVQATREDKFSFGLWTVGWQARDAFGDATRTALDPVEAVHKLAEIG...

In [4]:
df = df[df["macromoleculeType"] == "Protein"]

In [6]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

df = df[
    df["sequence"].apply(
        lambda x: isinstance(x, str)
        and len(x) > 0
        and set(x.upper()).issubset(VALID_AA)
    )
]

In [7]:
df["sequence"] = df["sequence"].str.upper().str.strip()

In [8]:
df["sequence_length"] = df["sequence"].str.len()

In [9]:
df["sequence_length"].describe()

count    337309.000000
mean        267.771901
std         193.192662
min           2.000000
25%         143.000000
50%         232.000000
75%         345.000000
max        5037.000000
Name: sequence_length, dtype: float64

In [10]:
df['sequence_length_normalized'] = np.log1p(df['sequence_length'])

In [11]:
df["sequence_length_normalized"].describe()

count    337309.000000
mean          5.330101
std           0.830694
min           1.098612
25%           4.969813
50%           5.451038
75%           5.846439
max           8.524764
Name: sequence_length_normalized, dtype: float64

In [13]:
metadata = pd.read_csv('Dataset/pdb_data_no_dups.csv')

In [14]:
merged = df.merge(
    metadata[["structureId", "classification"]],
    on="structureId",
    how="inner"
)

In [15]:
merged["classification"].value_counts()

classification
HYDROLASE                                   45725
TRANSFERASE                                 36073
OXIDOREDUCTASE                              33670
IMMUNE SYSTEM                               15459
LYASE                                       11574
                                            ...  
cell cycle/inhibitor                            1
Oxidoreductase, hydrolase                       1
OXIDOREDUCTASE(CHNH(D)-NAD+ OR NADP+(A))        1
ELECTRON TRANSPORT(IRON)                        1
antimicrobial                                   1
Name: count, Length: 4401, dtype: int64

In [16]:
top20 = merged["classification"].value_counts().head(20).index

df20 = merged[
    merged["classification"].isin(top20)
].copy()

In [17]:
df20["classification"].value_counts()

classification
HYDROLASE                                45725
TRANSFERASE                              36073
OXIDOREDUCTASE                           33670
IMMUNE SYSTEM                            15459
LYASE                                    11574
HYDROLASE/HYDROLASE INHIBITOR            10376
TRANSCRIPTION                             8816
VIRAL PROTEIN                             8325
TRANSPORT PROTEIN                         8291
VIRUS                                     6954
SIGNALING PROTEIN                         6365
ISOMERASE                                 6323
LIGASE                                    4910
PROTEIN BINDING                           4821
MEMBRANE PROTEIN                          4688
STRUCTURAL PROTEIN                        4284
CHAPERONE                                 4112
STRUCTURAL GENOMICS, UNKNOWN FUNCTION     3545
SUGAR BINDING PROTEIN                     3374
DNA BINDING PROTEIN                       3179
Name: count, dtype: int64

In [18]:
df20["sequence"].duplicated().sum()

np.int64(172243)

In [19]:
df20.duplicated(
    subset=["structureId", "chainId"]
).sum()

np.int64(777)

In [21]:
df.isnull().sum()

structureId                   0
chainId                       2
sequence                      0
residueCount                  0
macromoleculeType             0
sequence_length               0
sequence_length_normalized    0
dtype: int64